# JupyterLab Firefox Launcher - Deployment Debugging

This notebook will help you debug why the Firefox launcher isn't working on your JupyterHub deployment.

**Error:** 404 Not Found when accessing `/hub/firefox-desktop/`

**URL:** https://bdxbdxbdx-0d317c8b-1cfe-423e-a518-57f97fd50c6e.vantagecompute.ai/hub/firefox-desktop/

## Debugging Steps

We'll check:
1. Extension installation status
2. Server-proxy configuration
3. System dependencies (Xpra, Firefox)
4. JupyterHub configuration
5. Log files and error messages

## 1. Check Extension Installation

First, let's verify if the Firefox launcher extension is properly installed.

In [ ]:
# Check if the Firefox launcher package is installed
import subprocess
import sys

def check_package_installed(package_name):
    try:
        result = subprocess.run([sys.executable, '-m', 'pip', 'show', package_name], 
                              capture_output=True, text=True)
        if result.returncode == 0:
            print(f"✅ {package_name} is installed")
            print(result.stdout)
            return True
        else:
            print(f"❌ {package_name} is NOT installed")
            return False
    except Exception as e:
        print(f"❌ Error checking {package_name}: {e}")
        return False

# Check our extension
check_package_installed("jupyterlab-firefox-launcher")

In [ ]:
# Check JupyterLab extensions
try:
    result = subprocess.run(['jupyter', 'labextension', 'list'], 
                          capture_output=True, text=True)
    print("📋 JupyterLab Extensions:")
    print(result.stdout)
    
    if "jupyterlab-firefox-launcher" in result.stdout:
        print("✅ Firefox launcher frontend extension is installed")
    else:
        print("❌ Firefox launcher frontend extension is NOT found")
        
except Exception as e:
    print(f"❌ Error checking JupyterLab extensions: {e}")

## 2. Check Server Proxy Configuration

Now let's verify if jupyter-server-proxy is working and our Firefox desktop endpoint is registered.

In [ ]:
# Check jupyter-server-proxy
check_package_installed("jupyter-server-proxy")

print("\n" + "="*50)

# Check registered server proxy servers
try:
    import pkg_resources
    
    print("📋 Registered jupyter-server-proxy servers:")
    
    # Get all entry points for jupyter_serverproxy_servers
    entry_points = pkg_resources.iter_entry_points('jupyter_serverproxy_servers')
    
    found_firefox = False
    for entry_point in entry_points:
        print(f"  • {entry_point.name}: {entry_point.module_name}:{entry_point.attrs[0]}")
        if entry_point.name == "firefox-desktop":
            found_firefox = True
            print("    ✅ Firefox desktop server found!")
    
    if not found_firefox:
        print("❌ firefox-desktop server proxy entry point NOT found")
        
except Exception as e:
    print(f"❌ Error checking server proxy entries: {e}")

In [ ]:
# Test the server proxy function directly
try:
    from jupyterlab_firefox_launcher.server_proxy import setup_firefox_desktop
    
    print("✅ Successfully imported setup_firefox_desktop function")
    
    # Try to call the setup function
    config = setup_firefox_desktop()
    print("✅ Server proxy configuration created successfully")
    print(f"📋 Config: {config}")
    
except ImportError as e:
    print(f"❌ Cannot import setup_firefox_desktop: {e}")
    print("   This suggests the package is not installed or not in the Python path")
    
except Exception as e:
    print(f"❌ Error calling setup_firefox_desktop: {e}")
    print(f"   This might indicate missing system dependencies: {e}")

## 3. Check System Dependencies

The Firefox launcher requires Xpra and Firefox to be installed on the system.

In [ ]:
# Check system dependencies
import shutil
import os

def check_command(command):
    """Check if a command is available in the system PATH"""
    path = shutil.which(command)
    if path:
        print(f"✅ {command} found at: {path}")
        return True
    else:
        print(f"❌ {command} NOT found in PATH")
        return False

print("🔍 Checking system dependencies:")
print()

# Check for Xpra
xpra_available = check_command("xpra")

# Check for Firefox
firefox_available = check_command("firefox")

# Check for display server (for headless operation)
print()
display = os.environ.get('DISPLAY')
if display:
    print(f"📺 DISPLAY environment variable: {display}")
else:
    print("❌ No DISPLAY environment variable set")
    print("   This is expected for headless servers")

# Check for Xpra HTML5 support
if xpra_available:
    try:
        result = subprocess.run(['xpra', '--help'], capture_output=True, text=True)
        if 'html' in result.stdout.lower():
            print("✅ Xpra appears to have HTML5 support")
        else:
            print("⚠️  Cannot confirm Xpra HTML5 support")
    except Exception as e:
        print(f"⚠️  Error checking Xpra features: {e}")

## 4. Check JupyterHub Configuration

Let's examine the current Jupyter server configuration and URL structure.

In [ ]:
# Check current Jupyter server info
import json
import requests
from urllib.parse import urlparse

# Get current server information
try:
    # This should work in a JupyterLab/Notebook environment
    import IPython
    connection_file = IPython.get_ipython().config['IPKernelApp']['connection_file']
    print(f"📋 Kernel connection file: {connection_file}")
except:
    print("ℹ️  Not running in IPython kernel or cannot access connection info")

print()

# Check environment variables related to Jupyter
jupyter_env_vars = ['JUPYTERHUB_BASE_URL', 'JUPYTERHUB_SERVICE_PREFIX', 
                   'JUPYTER_SERVER_URL', 'JUPYTERHUB_API_URL']

for var in jupyter_env_vars:
    value = os.environ.get(var)
    if value:
        print(f"🔧 {var}: {value}")
    else:
        print(f"❌ {var}: not set")

print()

# Try to detect the current server URL
try:
    # Get the current URL (this works in JupyterLab)
    from IPython.display import Javascript, display
    
    # This will help us understand the URL structure
    js_code = """
    console.log('Current window location:', window.location.href);
    console.log('Base URL:', window.location.origin);
    element.text('Current URL: ' + window.location.href);
    """
    
    display(Javascript(js_code))
    
except Exception as e:
    print(f"Cannot detect current URL: {e}")

# Check if we're running in JupyterHub
hub_base = os.environ.get('JUPYTERHUB_BASE_URL', '')
if hub_base:
    print(f"✅ Running in JupyterHub with base URL: {hub_base}")
else:
    print("ℹ️  Not running in JupyterHub or JUPYTERHUB_BASE_URL not set")

## 5. Troubleshooting Steps

Based on the checks above, here are the most likely issues and solutions:

### Common Issues and Solutions:

**1. Extension Not Installed**
```bash
# Install the extension on the JupyterHub server
pip install jupyterlab-firefox-launcher
# Or from wheel
pip install jupyterlab_firefox_launcher-0.7.1-py3-none-any.whl
```

**2. Missing System Dependencies**
```bash
# On Ubuntu/Debian
sudo apt-get update
sudo apt-get install xpra xpra-html5 firefox

# On RHEL/CentOS
sudo yum install xpra python3-xpra-html5 firefox

# On Conda
conda install -c conda-forge xpra firefox
```

**3. JupyterHub Configuration Issue**
- The extension might not be installed in the correct Python environment
- JupyterHub might be using a different Python environment than where the extension is installed
- Server might need to be restarted after installation

**4. URL Path Issues**
- Check if the URL should be `/user/{username}/firefox-desktop/` instead of `/hub/firefox-desktop/`
- Verify JupyterHub base URL configuration

### Manual Installation Commands:

Run these on the JupyterHub server:

In [ ]:
# 1. Install system dependencies
sudo apt-get update
sudo apt-get install xpra xpra-html5 firefox

# 2. Install the Python package (use the same Python environment as JupyterHub)
pip install jupyterlab-firefox-launcher

# 3. Verify installation
pip show jupyterlab-firefox-launcher
jupyter labextension list

# 4. Restart JupyterHub
sudo systemctl restart jupyterhub
# or however your JupyterHub is managed

# 5. Test the endpoint manually
curl -I http://localhost:8000/hub/firefox-desktop/

## ✅ SOLUTION FOUND!

**The diagnostic shows everything is installed correctly:**
- ✅ Extension installed: `jupyterlab-firefox-launcher v0.7.1`
- ✅ Server proxy registered: `firefox-desktop` endpoint found
- ✅ System dependencies: Xpra and Firefox both installed with HTML5 support
- ✅ Configuration working: Server proxy config generated successfully

**The issue is the URL path!**

From the environment variables, we can see:
- `JUPYTERHUB_SERVICE_PREFIX: /user/ubuntu/a/`

## 🎯 **Correct URL:**

Instead of: 
```
https://bdxbdxbdx-0d317c8b-1cfe-423e-a518-57f97fd50c6e.vantagecompute.ai/hub/firefox-desktop/
```

**Use this URL:**
```
https://bdxbdxbdx-0d317c8b-1cfe-423e-a518-57f97fd50c6e.vantagecompute.ai/user/ubuntu/a/firefox-desktop/
```

**Or try the launcher icon in JupyterLab** - it should appear in the "Other" section and will automatically use the correct URL path.

## 🚀 **Next Steps:**
1. Click the **Firefox Desktop** icon in the JupyterLab launcher
2. Or manually navigate to the corrected URL above
3. You should see the Firefox desktop loading via Xpra HTML5

**Everything is properly configured - it was just a URL routing issue!**